In [ ]:
[susie_bias_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}_summary.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    
    # -----------------------------------------------------------------------------
    # Helpers
    #   true_set_num     : # of CSs that intersect any true causal (true positives,
    #                      at the CS level — denominator-friendly for FDR)
    #   discover_var_num : # of true causals captured anywhere in the CS union
    #                      (numerator-friendly for power / discovery rate)
    # -----------------------------------------------------------------------------
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) {
        return(0L)
      }
      true_set <- integer(length(cs))
      for (i in seq_along(cs)) {
        true_set[i] <- as.integer(length(intersect(variant, cs[[i]])) > 0)
      }
      sum(true_set)
    }
    
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) {
        return(0L)
      }
      length(intersect(variant, unlist(cs)))
    }
    
    # -----------------------------------------------------------------------------
    # Aggregate across all simulation outputs in `folder`.
    # Each rds is the `output` list from the finemap script:
    #   $result_list  : named list of susie_rss fits (bare fit objects)
    #   $true_vars    : indices of true causals (gtex / shared_vars order)
    # -----------------------------------------------------------------------------
    path  <- ${folder:r}
    files <- list.files(path, pattern = "sample_.*\\.rds$",
                        recursive = TRUE, full.names = TRUE)
    
    tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      ncausal <- as.numeric(sub(".*ncausal_([0-9.]+).*", "\\1", f))
    
      # Build one tibble row from a fit-like object: either a full susie_rss
      # fit or a SER sub-model stored under R_finite_diagnostics$ser_model.
      # `diag` is passed explicitly since the SER model has no diagnostics of
      # its own (fields fall back to NA).
      row_from_fit <- function(fit, label, diag) {
        cs <- fit$sets$cs
        tibble(
          input_file       = f,
          method           = label,
          ncausal          = ncausal,
          true_var_number  = discover_var_num(variant, cs),
          true_set_number  = true_set_num(variant, cs),
          total_var_number = length(variant),
          total_set_number = length(cs),
          total_set_length = length(unlist(cs)),
    
          B_corrected      = fit$B_corrected %||% NA_real_,
          lambda_bias      = fit$lambda_bias %||% NA_real_,
    
          # ---- artifact side (re-thresholdable scalar + flag) ----
          Q_art              = diag$Q_art              %||% NA_real_,
          artifact_evaluable = diag$artifact_evaluable %||% NA,
    
          # ---- sensitivity side (raw scalar = max over CS, + flag) ----
          D_CS_max         = if (!is.null(diag$bf_attenuation$cs_max))
                               suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE))
                             else NA_real_,
    
          # ---- combined ----
          R_reliability_flag = diag$R_reliability_flag %||% NA,
          prior_flag =  diag$prior_variance_flag %||% NA
        )
      }
    
      susie_methods <- names(res$result_list)  # dynamically read available methods
    
      map_dfr(susie_methods, function(m) {
        sus  <- res[["result_list"]][[m]]
        diag <- sus$R_finite_diagnostics
    
        rows <- row_from_fit(sus, m, diag)
    
        # If this method carries a SER sub-model, emit it as "<method>_SER".
        if (!is.null(diag$ser_model)) {
          ser  <- diag$ser_model
          rows <- bind_rows(
            rows,
            row_from_fit(ser, paste0(m, "_SER"), ser$R_finite_diagnostics)
          )
        }
        rows
      })
    })
    saveRDS(tb, ${_output:r})

In [ ]:
[susie_minor_effect_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}_summary_minor_effect.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    
    # -----------------------------------------------------------------------------
    # Helpers
    #   true_set_num     : # of CSs that intersect any true causal (true positives,
    #                      at the CS level — denominator-friendly for FDR)
    #   discover_var_num : # of true causals captured anywhere in the CS union
    #                      (numerator-friendly for power / discovery rate)
    # -----------------------------------------------------------------------------
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) {
        return(0L)
      }
      true_set <- integer(length(cs))
      for (i in seq_along(cs)) {
        true_set[i] <- as.integer(length(intersect(variant, cs[[i]])) > 0)
      }
      sum(true_set)
    }
    
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) {
        return(0L)
      }
      length(intersect(variant, unlist(cs)))
    }
    
    # -----------------------------------------------------------------------------
    # Aggregate across all simulation outputs in `folder`.
    # Each rds is the `output` list from the finemap script:
    #   $result_list  : named list of susie_rss fits (bare fit objects)
    #   $true_vars    : indices of true causals (gtex / shared_vars order)
    # -----------------------------------------------------------------------------
    path  <- ${folder:r}
    files <- list.files(path, pattern = "sample_.*\\.rds$",
                        recursive = TRUE, full.names = TRUE)
    
    tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      ncausal <- as.numeric(sub(".*ncausal_([0-9.]+).*", "\\1", f))
      
      susie_methods <- names(res$result_list)  # dynamically read available methods
    
      map_dfr(susie_methods, function(m) {
        sus <- res[["result_list"]][[m]]
        cs  <- sus$sets$cs
        diag <- sus$R_finite_diagnostics
       tibble(
          input_file       = f,
          method           = m,
          ncausal          = ncausal,
          true_var_number  = discover_var_num(variant, cs),
          true_set_number  = true_set_num(variant, cs),
          total_var_number = length(variant),
          total_set_number = length(cs),
          total_set_length = length(unlist(cs)),
           
        
          B_corrected      = sus$B_corrected %||% NA_real_,
          lambda_bias      = sus$lambda_bias %||% NA_real_,
        found_rank1 = if (length(variant) >= 1) as.integer(variant[1] %in% unlist(cs)) else NA_integer_,
        found_rank2 = if (length(variant) >= 2) as.integer(variant[2] %in% unlist(cs)) else NA_integer_,
        found_rank3 = if (length(variant) >= 3) as.integer(variant[3] %in% unlist(cs)) else NA_integer_,
        
          # ---- artifact side (re-thresholdable scalar + flag) ----
          Q_art            = diag$Q_art            %||% NA_real_,
          artifact_evaluable = diag$artifact_evaluable %||% NA,
        
          # ---- sensitivity side (raw scalar = max over CS, + flag) ----
          D_CS_max         = if (!is.null(diag$bf_attenuation$cs_max))
                               suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE))
                             else NA_real_,        
          # ---- combined ----
          R_reliability_flag = diag$R_reliability_flag %||% NA,
            prior_flag =  diag$prior_variance_flag %||% NA
        )
      })
    })

    saveRDS(tb, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J susie_minor_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb" susie_bias_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Hetero_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J susie_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb" susie_bias_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/GWAS_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J susie_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb" susie_bias_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Hetero_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J minor_susie_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/minor_summary.%J.out
#BSUB -e WORK_DIR/summary/minor_summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb" susie_minor_effect_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="minor_summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
res = readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Hetero_fine/result/h2g_0.15/causal_2/sample_0_ncausal_2_h2g_0.15_multi_ref.rds")

sus = res$result_list$ADSP_mix$R_finite_diagnostics

## If using current version of susieR github version
sus$R_reliability_flag # True means SER

names(sus)

## if susieR not updated (using old thresholds)

sus$Q_art > 0.1
max(sus$bf_attenuation$cs_max) > 30 ## either one True then SER. 

## Check the European ancestry GTEx simulations

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_EUR_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J susie_minor_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb" susie_bias_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

## Iterations: SuSiE-RSS vs SuSiE-rRSS

`[niter_summary]` reads result files for one setting or the subfolder selected by `--part`. For each SuSiE fit, including fits nested up to three levels deep, it records `niter`, `converged`, L, p, number of credible sets, and for fits with `R_finite_diagnostics` the reliability flag. The summary includes SER fits (L = 1), which the plotting notebook omits. See `Data_Processing/Figure_1/Supp.Niter_SuSiE_RSS_vs_rRSS`.

The iteration counts reflect different stopping rules. With `R_finite` or `R_mismatch`, susieR checks convergence using PIPs. SuSiE-RSS checks convergence using the ELBO.

Run for the allele-flip (`Fine_Flip` from `02`, `Fine_Flip_rRSS` from `25`), GTEx population-mismatch (`Gtex_fine` from `09`) and UKB sketch (`UKBB_B_finemap` from `05`, Fig 2B) simulations, one job per folder (submit cell below).


In [ ]:
[niter_summary]
# Iterations of every SuSiE fit stored in one setting's result files (any slot, e.g. result_list$kriging_res).
parameter: folder = path
# the setting's result folder
parameter: name = str
# setting name (the setting column and the output file)
parameter: part = ""
# part of the setting (subfolder) when it is split over jobs; added to the output file name
parameter: cwd = path("output")
parameter: walltime = "24h"
parameter: mem = "30G"
parameter: numThreads = 4
parameter: container = ""
output: f'{cwd:a}/{name}{"_" + part if part else ""}_niter.rds'
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{name}_{part}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(parallel)
    `%||%` <- function(a, b) if (is.null(a)) b else a

    files <- list.files(${folder:r}, pattern = "\\.rds$", recursive = TRUE, full.names = TRUE)

    ## every SuSiE fit in a result file, searched up to 3 levels down (e.g. result_list$kriging_res);
    ## a fit is not searched inside, so the SER model in R_finite_diagnostics is not picked up
    is_fit <- function(x) is.list(x) && !is.null(x$alpha) && !is.null(x$pip)
    find_fits <- function(x, path = character(0), depth = 0) {
      if (is_fit(x)) return(setNames(list(x), paste(path, collapse = "$")))
      if (!is.list(x) || is.data.frame(x) || depth >= 3 || is.null(names(x))) return(list())
      out <- list()
      for (nm in names(x)) if (nzchar(nm)) out <- c(out, find_fits(x[[nm]], c(path, nm), depth + 1))
      out
    }

    one_file <- function(f) tryCatch({
      fits <- find_fits(readRDS(f))
      if (!length(fits)) return(NULL)
      bind_rows(Map(function(m, slot) {             # m = one fit (not "fit": tibble() would see the column)
        d <- m$R_finite_diagnostics
        tibble(file = f, slot = slot, fit = sub(".*\\$", "", slot),
               niter = as.integer(m$niter %||% NA), converged = as.logical(m$converged %||% NA),
               L = nrow(m$alpha), p = ncol(m$alpha), n_cs = length(m$sets$cs),
               has_R_diag = !is.null(d), R_reliability_flag = as.logical(d$R_reliability_flag %||% NA))
      }, fits, names(fits)))
    }, error = function(e) { message("FAILED ", f, ": ", conditionMessage(e)); NULL })

    out <- bind_rows(mclapply(files, one_file, mc.cores = ${numThreads})) %>% mutate(setting = "${name}")
    saveRDS(out, ${_output:r})
    cat(sprintf("%s: %d files, %d fits, niter missing for %d\n", "${name}", length(files), nrow(out), sum(is.na(out$niter))))
    print(as.data.frame(count(out, fit, L, has_R_diag)))

In [ ]:
# allele flip (02: Fine_Flip, 25: Fine_Flip_rRSS), GTEx population mismatch (09: Gtex_fine), UKB sketch (05: UKBB_B_finemap); one job per folder
# Fine_Flip, Fine_Flip_rRSS and Gtex_fine are done; add them to `settings` to rerun
# output: 2026_output/niter_summary/<setting>_niter.rds
base="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
out_dir="${base}/niter_summary"
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb"
mkdir -p ${out_dir}/log ${out_dir}/code
cd ${out_dir}/code

settings="UKBB_B_finemap"
for setting in ${settings}; do
  cat << EOF > niter_${setting}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J niter_${setting}
#BSUB -n 8
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=8000]
#BSUB -W 48:00
#BSUB -o ${out_dir}/log/niter_${setting}.%J.out
#BSUB -e ${out_dir}/log/niter_${setting}.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH:/usr/bin:/bin
sos run "${nb}" niter_summary --folder ${base}/${setting}/result --name ${setting} --cwd ${out_dir} --numThreads 8 --walltime 48h --mem 60G
EOF
  sb niter_${setting}.sh
done